# 🔬 MetaboNet Raw Data — Forensic EDA

**Dataset:** `metabonet_public_train.parquet`  
**Objective:** Decode, validate, and profile the raw Parquet file before Glucose Forecasting.  
**Stack:** `polars` (speed) + `plotly` (interactive charts)  

---

In [ ]:
# ── 0. Environment Setup ──────────────────────────────────────────────
!pip install polars plotly pyarrow -q

import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import os
import warnings
warnings.filterwarnings('ignore')

# ── Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

print(f"Polars version : {pl.__version__}")
print(f"Notebook run at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# ── File Paths ────────────────────────────────────────────────────────
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"

PARQUET_PATH      = f"{BASE_PATH}/raw/metabonet_public_train.parquet"
PARQUET_PATH_TEST = f"{BASE_PATH}/raw/metabonet_public_test.parquet"

print(f"📁 Train: {PARQUET_PATH}  — exists: {os.path.exists(PARQUET_PATH)}")
print(f"📁 Test : {PARQUET_PATH_TEST} — exists: {os.path.exists(PARQUET_PATH_TEST)}")

---
## STEP 1 — The Rosetta Stone: Smart Decoding 🔑
Many columns are stored as `List[Int]` (ASCII codes).  
We decode them back to human-readable strings.

In [ ]:
# ── 1.1 Load the raw parquet ──────────────────────────────────────────
df_raw = pl.read_parquet(PARQUET_PATH)

print(f"Shape  : {df_raw.shape}")
print(f"Columns: {df_raw.columns}")
print(f"\nSchema:")
for col, dtype in df_raw.schema.items():
    print(f"  {col:30s} → {dtype}")

In [ ]:
# ── 1.2 Identify encoded (List[Int]) columns automatically ───────────
encoded_cols = [
    col for col, dtype in df_raw.schema.items()
    if str(dtype).startswith("List") and (
        "Int" in str(dtype) or "UInt" in str(dtype) or "i64" in str(dtype).lower()
        or "u8" in str(dtype).lower() or "i8" in str(dtype).lower()
        or "i32" in str(dtype).lower() or "u32" in str(dtype).lower()
    )
]

print(f"Found {len(encoded_cols)} encoded columns:")
for c in encoded_cols:
    sample = df_raw[c].drop_nulls().head(1).to_list()
    preview = sample[0] if sample else "[NULL]"
    print(f"  {c:30s} → sample: {preview}")

In [ ]:
# ── 1.3 Decoding Function ────────────────────────────────────────────
def decode_ascii_list(series: pl.Series) -> pl.Series:
    """
    Decode a Polars Series of List[Int] (ASCII codes) into Utf8 strings.
    Handles nulls gracefully.
    """
    def _decode_one(lst):
        if lst is None:
            return None
        try:
            return "".join(chr(i) for i in lst if i is not None)
        except (TypeError, ValueError):
            return None

    return pl.Series(
        name=series.name,
        values=[_decode_one(v) for v in series.to_list()],
        dtype=pl.Utf8,
    )


# ── 1.4 Apply decoding to all identified columns ─────────────────────
df = df_raw.clone()

for col in encoded_cols:
    print(f"Decoding: {col} ...", end=" ")
    df = df.with_columns(decode_ascii_list(df[col]).alias(col))
    print("✅")

print(f"\nAll {len(encoded_cols)} columns decoded successfully.")

In [ ]:
# ── 1.5 Sanity Check ─────────────────────────────────────────────────
print("=" * 60)
print("DECODING SANITY CHECK")
print("=" * 60)

for col in encoded_cols:
    unique_vals = df[col].drop_nulls().unique().sort().head(10).to_list()
    print(f"\n{col}:")
    print(f"  Unique values (first 10): {unique_vals}")
    print(f"  Null count: {df[col].null_count()} / {len(df)}")

print("\n" + "=" * 60)
print("First 5 rows of decoded data:")
df.select(encoded_cols).head(5)

---
## STEP 2 — Temporal Parsing ⏱️
Convert the `date` column from Int64 (Nanosecond UNIX timestamp) to Datetime.

In [ ]:
# ── 2.1 Inspect raw date values ──────────────────────────────────────
print("Raw `date` column dtype:", df["date"].dtype)
print("Sample values:")
print(df["date"].head(5).to_list())

In [ ]:
# ── 2.2 Convert to Datetime ──────────────────────────────────────────
# Try nanoseconds first. If years come out as 1970, switch to microseconds.

# Attempt 1: Nanoseconds → Datetime
df = df.with_columns(
    pl.from_epoch(pl.col("date"), time_unit="ns").alias("date_parsed_ns")
)

sample_ns = df["date_parsed_ns"].head(3).to_list()
print(f"Nanosecond interpretation: {sample_ns}")

# Check: do years make sense?
min_year_ns = df["date_parsed_ns"].min().year if df["date_parsed_ns"].min() is not None else 0
max_year_ns = df["date_parsed_ns"].max().year if df["date_parsed_ns"].max() is not None else 0
print(f"Year range (ns): {min_year_ns} – {max_year_ns}")

if min_year_ns >= 2015 and max_year_ns <= 2030:
    print("✅ Nanosecond precision confirmed.")
    df = df.with_columns(pl.col("date_parsed_ns").alias("datetime")).drop("date_parsed_ns")
else:
    print("⚠️  Nanoseconds produced bad years. Trying microseconds...")
    df = df.with_columns(
        pl.from_epoch(pl.col("date"), time_unit="us").alias("date_parsed_us")
    )
    sample_us = df["date_parsed_us"].head(3).to_list()
    print(f"  Microsecond interpretation: {sample_us}")
    min_year_us = df["date_parsed_us"].min().year if df["date_parsed_us"].min() is not None else 0
    max_year_us = df["date_parsed_us"].max().year if df["date_parsed_us"].max() is not None else 0
    print(f"  Year range (us): {min_year_us} – {max_year_us}")
    
    if min_year_us >= 2015 and max_year_us <= 2030:
        print("  ✅ Microsecond precision confirmed.")
        df = df.with_columns(pl.col("date_parsed_us").alias("datetime")).drop(["date_parsed_ns", "date_parsed_us"])
    else:
        print("  ⚠️  Trying milliseconds as last resort...")
        df = df.with_columns(
            pl.from_epoch(pl.col("date"), time_unit="ms").alias("date_parsed_ms")
        )
        df = df.with_columns(pl.col("date_parsed_ms").alias("datetime")).drop(["date_parsed_ns", "date_parsed_us", "date_parsed_ms"])
        print(f"  Millisecond interpretation: {df['datetime'].head(3).to_list()}")

print(f"\nFinal datetime range: {df['datetime'].min()} → {df['datetime'].max()}")

---
## STEP 3 — Structural Integrity (Group by Decoded ID) 📊
How many unique patients? Is the data balanced?

In [ ]:
# ── 3.1 Patient Count ────────────────────────────────────────────────
n_patients = df["id"].n_unique()
print(f"Total unique patients (decoded `id`): {n_patients}")
print(f"Total rows: {len(df):,}")
print(f"Avg rows per patient: {len(df) / n_patients:,.0f}")

In [ ]:
# ── 3.2 Rows per Patient ─────────────────────────────────────────────
patient_counts = (
    df.group_by("id")
    .agg(pl.len().alias("row_count"))
    .sort("row_count", descending=True)
)

print("Top 10 patients by data volume:")
print(patient_counts.head(10))
print(f"\nBottom 5 patients:")
print(patient_counts.tail(5))
print(f"\nRow count stats:")
print(patient_counts["row_count"].describe())

In [ ]:
# ── 3.3 Bar Chart: Rows per Patient ──────────────────────────────────
pc_pd = patient_counts.to_pandas()

fig = px.bar(
    pc_pd.sort_values("row_count", ascending=False),
    x="id",
    y="row_count",
    title="📊 Rows per Patient (Sorted Descending)",
    labels={"id": "Patient ID", "row_count": "Number of Rows"},
    color="row_count",
    color_continuous_scale="Viridis",
)
fig.update_layout(
    template="plotly_dark",
    xaxis_tickangle=-45,
    height=500,
    font=dict(size=12),
)
fig.show()

In [ ]:
# ── 3.4 Demographics Breakdown ───────────────────────────────────────
demo_cols = [c for c in ["gender", "ethnicity", "cgm_device", "insulin_delivery_device"] if c in df.columns]

for col in demo_cols:
    print(f"\n{'─'*50}")
    print(f"Column: {col}")
    val_counts = df[col].value_counts().sort("count", descending=True)
    print(val_counts)

# Pie chart for gender if available
if "gender" in df.columns:
    gender_df = df["gender"].value_counts().to_pandas()
    fig = px.pie(
        gender_df, names="gender", values="count",
        title="⚧ Gender Distribution",
        color_discrete_sequence=px.colors.qualitative.Set2,
    )
    fig.update_layout(template="plotly_dark")
    fig.show()

---
## STEP 4 — Time-Series Health: The Gap Analysis 🕰️
Detect sampling frequency, gaps, and session continuity.

In [ ]:
# ── 4.1 Sort by patient + datetime ───────────────────────────────────
df = df.sort(["id", "datetime"])
print("Data sorted by [id, datetime]. ✅")

In [ ]:
# ── 4.2 Calculate time differences (partitioned by patient) ──────────
df = df.with_columns(
    (pl.col("datetime") - pl.col("datetime").shift(1).over("id"))
    .alias("time_diff")
)

# Convert to minutes for easier analysis
df = df.with_columns(
    (pl.col("time_diff").dt.total_seconds() / 60.0).alias("time_diff_min")
)

print("time_diff computed. Sample:")
df.select(["id", "datetime", "time_diff", "time_diff_min"]).head(10)

In [ ]:
# ── 4.3 Frequency Report ─────────────────────────────────────────────
valid_diffs = df.filter(pl.col("time_diff_min").is_not_null())

total_gaps = len(valid_diffs)
exact_5min = valid_diffs.filter(pl.col("time_diff_min") == 5.0).height
near_5min  = valid_diffs.filter(
    (pl.col("time_diff_min") >= 4.5) & (pl.col("time_diff_min") <= 5.5)
).height

pct_exact  = (exact_5min / total_gaps) * 100
pct_near   = (near_5min / total_gaps) * 100

print("╔══════════════════════════════════════════════════╗")
print("║         ⏱️  SAMPLING FREQUENCY REPORT           ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  Total time gaps analyzed : {total_gaps:>15,}    ║")
print(f"║  Exactly 5 min           : {exact_5min:>15,}    ║")
print(f"║  Pct exactly 5 min       : {pct_exact:>14.2f}%    ║")
print(f"║  Near 5 min (±30s)       : {near_5min:>15,}    ║")
print(f"║  Pct near 5 min          : {pct_near:>14.2f}%    ║")
print("╚══════════════════════════════════════════════════╝")

print("\nTime gap statistics (minutes):")
print(valid_diffs["time_diff_min"].describe())

In [ ]:
# ── 4.4 Session Detection ────────────────────────────────────────────
# A new session starts when gap > 15 minutes
SESSION_GAP_THRESHOLD = 15  # minutes

df = df.with_columns(
    (
        (pl.col("time_diff_min") > SESSION_GAP_THRESHOLD)
        | pl.col("time_diff_min").is_null()
    )
    .cast(pl.Int32)
    .cum_sum()
    .over("id")
    .alias("session_id")
)

session_counts = (
    df.group_by("id")
    .agg(pl.col("session_id").max().alias("n_sessions"))
    .sort("n_sessions", descending=True)
)

total_sessions = session_counts["n_sessions"].sum()
print(f"Total sessions across all patients: {total_sessions:,}")
print(f"Average sessions per patient: {session_counts['n_sessions'].mean():.1f}")
print(f"Max sessions (single patient): {session_counts['n_sessions'].max()}")
print(f"\nSession counts per patient (top 10):")
print(session_counts.head(10))

In [ ]:
# ── 4.5 Gap Histogram (excluding standard 5-min gaps) ────────────────
outlier_gaps = valid_diffs.filter(
    (pl.col("time_diff_min") < 4.5) | (pl.col("time_diff_min") > 5.5)
)

print(f"Non-standard gaps: {len(outlier_gaps):,} ({len(outlier_gaps)/total_gaps*100:.2f}% of all gaps)")

# Clip at 120 min for better visualization
outlier_pd = outlier_gaps.select(
    pl.col("time_diff_min").clip(0, 120).alias("gap_min")
).to_pandas()

fig = px.histogram(
    outlier_pd, x="gap_min", nbins=100,
    title="🔍 Distribution of Non-Standard Time Gaps (clipped at 120 min)",
    labels={"gap_min": "Gap (minutes)", "count": "Frequency"},
    color_discrete_sequence=["#FF6B6B"],
)
fig.update_layout(
    template="plotly_dark",
    height=450,
    bargap=0.05,
)
fig.show()

In [ ]:
# ── 4.6 Gap distribution table ───────────────────────────────────────
gap_buckets = valid_diffs.with_columns(
    pl.when(pl.col("time_diff_min") <= 5.5)
    .then(pl.lit("≤5.5 min (Normal)"))
    .when(pl.col("time_diff_min") <= 15)
    .then(pl.lit("5.5–15 min (Minor gap)"))
    .when(pl.col("time_diff_min") <= 60)
    .then(pl.lit("15–60 min (Session break)"))
    .when(pl.col("time_diff_min") <= 1440)
    .then(pl.lit("1–24 hours (Long gap)"))
    .otherwise(pl.lit(">24 hours (Day+ gap)"))
    .alias("gap_bucket")
)

bucket_counts = (
    gap_buckets.group_by("gap_bucket")
    .agg(pl.len().alias("count"))
    .with_columns((pl.col("count") / total_gaps * 100).round(2).alias("pct"))
    .sort("count", descending=True)
)

print("Gap Distribution by Bucket:")
print(bucket_counts)

---
## STEP 5 — Target & Physics Analysis 🧬
Inspect CGM (target), missing patterns, and visual trace.

In [ ]:
# ── 5.1 CGM Statistics ───────────────────────────────────────────────
if "CGM" in df.columns:
    cgm_col = "CGM"
elif "cgm" in df.columns:
    cgm_col = "cgm"
else:
    # Try to find it case-insensitively
    cgm_col = [c for c in df.columns if c.lower() == "cgm"]
    cgm_col = cgm_col[0] if cgm_col else None
    print(f"⚠️  CGM column not found! Available columns: {df.columns}")

if cgm_col:
    print(f"Using target column: `{cgm_col}`")
    print(f"\n{'═'*50}")
    print(f"CGM STATISTICS")
    print(f"{'═'*50}")
    cgm_stats = df[cgm_col].describe()
    print(cgm_stats)
    
    print(f"\nNull count: {df[cgm_col].null_count():,} / {len(df):,} ({df[cgm_col].null_count()/len(df)*100:.2f}%)")
    print(f"Min : {df[cgm_col].min()}")
    print(f"Max : {df[cgm_col].max()}")
    print(f"Mean: {df[cgm_col].mean():.2f}")
    print(f"Std : {df[cgm_col].std():.2f}")

In [ ]:
# ── 5.2 CGM Distribution ─────────────────────────────────────────────
if cgm_col:
    cgm_pd = df.select(pl.col(cgm_col).cast(pl.Float64)).to_pandas()

    fig = make_subplots(rows=1, cols=2, subplot_titles=("CGM Histogram", "CGM Box Plot"))

    fig.add_trace(
        go.Histogram(x=cgm_pd[cgm_col].dropna(), nbinsx=100,
                     marker_color="#4ECDC4", name="CGM"),
        row=1, col=1
    )
    fig.add_trace(
        go.Box(y=cgm_pd[cgm_col].dropna(), marker_color="#FF6B6B", name="CGM"),
        row=1, col=2
    )

    fig.update_layout(
        title="🩸 CGM Target Distribution",
        template="plotly_dark",
        height=450,
        showlegend=False,
    )
    fig.show()

In [ ]:
# ── 5.3 Missing Value Analysis ───────────────────────────────────────
print("╔══════════════════════════════════════════════════════════════╗")
print("║             📋  MISSING VALUE REPORT                       ║")
print("╠════════════════════════════╦═══════════╦═══════════════════╣")
print("║ Column                     ║ Null Count║ Null %            ║")
print("╠════════════════════════════╬═══════════╬═══════════════════╣")

null_report = []
for col_name in df.columns:
    n_null = df[col_name].null_count()
    pct_null = (n_null / len(df)) * 100
    null_report.append((col_name, n_null, pct_null))
    print(f"║ {col_name:26s} ║ {n_null:>9,} ║ {pct_null:>15.2f}%  ║")

print("╚════════════════════════════╩═══════════╩═══════════════════╝")

# Highlight columns that are mostly null (>80%)
mostly_null = [c for c, n, p in null_report if p > 80]
sparse_cols = [c for c, n, p in null_report if 30 < p <= 80]
usable_cols = [c for c, n, p in null_report if p <= 30]

print(f"\n🔴 Mostly empty (>80% null): {mostly_null}")
print(f"🟡 Sparse (30–80% null):     {sparse_cols}")
print(f"🟢 Usable (≤30% null):       {usable_cols}")

In [ ]:
# ── 5.4 Nullity Heatmap ──────────────────────────────────────────────
null_pcts = pl.DataFrame({
    "column": [c for c, _, _ in null_report],
    "null_pct": [p for _, _, p in null_report],
}).sort("null_pct", descending=True).to_pandas()

fig = px.bar(
    null_pcts, x="column", y="null_pct",
    title="🗑️ Null Percentage per Column",
    labels={"column": "Column", "null_pct": "Null %"},
    color="null_pct",
    color_continuous_scale="RdYlGn_r",
    range_color=[0, 100],
)
fig.add_hline(y=80, line_dash="dash", line_color="red",
              annotation_text="80% threshold")
fig.add_hline(y=30, line_dash="dash", line_color="orange",
              annotation_text="30% threshold")
fig.update_layout(
    template="plotly_dark",
    height=500,
    xaxis_tickangle=-45,
)
fig.show()

In [ ]:
# ── 5.5 Visual Inspection: CGM Trace of Patient with Most Data ──────
if cgm_col:
    # Find patient with most rows
    top_patient = patient_counts["id"][0]
    top_patient_n = patient_counts["row_count"][0]
    
    print(f"Plotting CGM trace for patient '{top_patient}' ({top_patient_n:,} rows)")
    
    trace_df = (
        df.filter(pl.col("id") == top_patient)
        .select(["datetime", cgm_col])
        .sort("datetime")
        .to_pandas()
    )
    
    fig = px.line(
        trace_df, x="datetime", y=cgm_col,
        title=f"🩸 Full CGM Trace — Patient '{top_patient}' ({top_patient_n:,} readings)",
        labels={"datetime": "Time", cgm_col: "CGM (mg/dL)"},
    )
    fig.update_traces(line=dict(width=0.8, color="#00D4AA"))
    
    # Add clinical reference lines
    fig.add_hline(y=70,  line_dash="dash", line_color="red",    annotation_text="Hypo (70)")
    fig.add_hline(y=180, line_dash="dash", line_color="orange", annotation_text="Hyper (180)")
    fig.add_hline(y=250, line_dash="dash", line_color="red",    annotation_text="Critical High (250)")
    
    fig.update_layout(
        template="plotly_dark",
        height=500,
        xaxis_rangeslider_visible=True,
    )
    fig.show()

In [ ]:
# ── 5.6 Zoomed 24h CGM Trace ─────────────────────────────────────────
if cgm_col:
    # Pick middle of the trace for a 24h window
    mid_idx = len(trace_df) // 2
    mid_time = trace_df["datetime"].iloc[mid_idx]
    start_window = mid_time
    end_window = mid_time + timedelta(hours=24)
    
    zoom_df = trace_df[
        (trace_df["datetime"] >= start_window) &
        (trace_df["datetime"] <= end_window)
    ]
    
    fig = px.line(
        zoom_df, x="datetime", y=cgm_col,
        title=f"🔎 24-Hour Zoom — Patient '{top_patient}'",
        labels={"datetime": "Time", cgm_col: "CGM (mg/dL)"},
    )
    fig.update_traces(line=dict(width=1.5, color="#FFD93D"), mode="lines+markers")
    fig.add_hline(y=70,  line_dash="dash", line_color="red")
    fig.add_hline(y=180, line_dash="dash", line_color="orange")
    fig.update_layout(template="plotly_dark", height=450)
    fig.show()

---
## SECTION 6 — The Verdict 📜
Final data quality report.

In [ ]:
# ── 6. THE VERDICT ───────────────────────────────────────────────────

print("")
print("╔══════════════════════════════════════════════════════════════════╗")
print("║                 📋 FINAL DATA QUALITY VERDICT                  ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║                                                                ║")

# 1. Decoding Status
decode_ok = all(df[c].dtype == pl.Utf8 for c in encoded_cols)
status_1  = "✅ SUCCESS" if decode_ok else "❌ FAILED"
print(f"║  1. Decoding Status:  {status_1:43s}  ║")
print(f"║     Decoded {len(encoded_cols)} columns from List[Int] → Utf8            ║")
print("║                                                                ║")

# 2. Data Quality / Frequency
freq_ok = pct_near > 90
status_2 = f"✅ CONSISTENT ({pct_near:.1f}% within 5±0.5 min)" if freq_ok else f"⚠️  INCONSISTENT ({pct_near:.1f}% within 5±0.5 min)"
print(f"║  2. Sampling Frequency:  {status_2:40s} ║")
print("║                                                                ║")

# 3. Features verdict
print("║  3. Feature Usability:                                        ║")
for c in usable_cols:
    if c not in ["time_diff", "time_diff_min", "session_id", "datetime"]:
        print(f"║     🟢 {c:56s}  ║")
for c in sparse_cols:
    print(f"║     🟡 {c:48s} (sparse) ║")
for c in mostly_null:
    print(f"║     🔴 {c:46s} (discard) ║")
print("║                                                                ║")

# 4. Ready for forecasting?
needs_resample = not freq_ok
has_target     = cgm_col is not None and df[cgm_col].null_count() / len(df) < 0.1

if has_target and not needs_resample:
    readiness = "✅ YES — Ready for glucose forecasting"
elif has_target and needs_resample:
    readiness = "⚠️  NEEDS RESAMPLING — Irregular frequency detected"
else:
    readiness = "❌ NO — Target column issues detected"

print(f"║  4. Ready for Forecasting?                                    ║")
print(f"║     {readiness:59s} ║")
print("║                                                                ║")
print(f"║  Dataset: {len(df):>12,} rows × {len(df.columns)} columns                     ║")
print(f"║  Patients: {n_patients:>10,}                                          ║")
print(f"║  Date Range: {str(df['datetime'].min())[:10]} → {str(df['datetime'].max())[:10]}                     ║")
print("║                                                                ║")
print("╚══════════════════════════════════════════════════════════════════╝")

In [ ]:
# ── Final Schema Summary ─────────────────────────────────────────────
print("\nFinal DataFrame Schema (after decoding):")
for col_name, dtype in df.schema.items():
    if col_name not in ["time_diff", "time_diff_min", "session_id", "datetime", "date"]:
        n = df[col_name].null_count()
        p = n / len(df) * 100
        print(f"  {col_name:30s} {str(dtype):15s}  nulls: {p:.1f}%")